# 04.2.1 - Supervised Learning: Decision Tree (K-Means QLDE)

## 1. Tujuan Analisis
Pada tahap sebelumnya, kita sudah melatih *Decision Tree* menggunakan data hasil K-Means standar. Sekarang, kita akan melakukan eksperimen yang sama, namun menggunakan **dataset yang berbeda**, yaitu hasil pelabelan dari algoritma **K-Means QLDE**.

**Kenapa kita melakukan ini?**
Kita ingin mengevaluasi seberapa baik pola kelompok pelanggan (*cluster*) yang dihasilkan oleh metode QLDE. Jika QLDE menghasilkan kelompok pelanggan yang logis dan terstruktur dengan baik, maka *Decision Tree* seharusnya bisa dengan mudah mengekstrak **Aturan Bisnis (Business Rules)** dari kelompok tersebut.

## 2. Alur Kerja
* **Input:** Dataset `hasildata_kmeans-qlde.csv` (11 Fitur Numerik dan 1 Target Cluster hasil QLDE).
* **Model:** *Decision Tree* dengan batas kedalaman 4 tingkat (`max_depth=4`).
* **Output:** Rapor performa model dan Teks Aturan Logika *If-Else*.

In [1]:
import joblib
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report

# 1. BACA DATA HASIL QLDE
filepath = '../data/Labeled/hasildata_kmeans-qlde.csv'
df = pd.read_csv(filepath)

# Pisahkan Fitur (Var1 - Var11) dan Target (Cluster hasil QLDE)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: 80% untuk Train (belajar), 20% untuk Test (ujian)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Training Model Decision Tree
Kita kembali menggunakan `max_depth=4`. Alasannya tetap sama: kita ingin aturan bisnis yang dihasilkan nanti cukup sederhana untuk dibaca oleh manusia (tim bisnis/marketing) dan bisa langsung diterapkan di lapangan tanpa terlalu banyak cabang kondisi.

In [2]:
# 2. LATIH MODEL DECISION TREE
model_dt_qlde = DecisionTreeClassifier(random_state=42, max_depth=4)
model_dt_qlde.fit(X_train, y_train)

# 3. EVALUASI MODEL
# Meminta model menebak data ujian (Test)
prediksi_dt_qlde = model_dt_qlde.predict(X_test)

# Menampilkan Rapor Performa
print("=== CLASSIFICATION REPORT: DECISION TREE (QLDE) ===\n")
print(classification_report(y_test, prediksi_dt_qlde, zero_division=0))

=== CLASSIFICATION REPORT: DECISION TREE (QLDE) ===

              precision    recall  f1-score   support

           0       0.92      0.92      0.92       150
           1       0.72      0.95      0.82       214
           2       0.90      0.85      0.87       125
           3       0.98      0.72      0.83        85
           4       0.91      0.83      0.87       199
           5       0.59      0.44      0.50        94

    accuracy                           0.83       867
   macro avg       0.84      0.78      0.80       867
weighted avg       0.83      0.83      0.82       867



### Analisis Performa (QLDE vs Standard)
Dari rapor di atas, kita mendapatkan tingkat akurasi sebesar **83%** (atau `0.83`). 
Angka akurasi keseluruhan ini ternyata **sama persis** dengan saat kita menggunakan dataset Standard K-Means. Namun, jika kita melihat lebih detail pada setiap kelas (terutama *Class 5*), nilainya sedikit turun. Ini menunjukkan bahwa meskipun performa umumnya setara, QLDE membentuk bentuk kelompok (cluster) yang sedikit lebih menantang untuk dipisahkan oleh batasan garis lurus (If-Else) pada *Decision Tree*.

## 4. Ekstraksi Aturan Bisnis QLDE
Mari kita lihat bagaimana bentuk aturan bisnis dari data QLDE ini. Aturan ini akan menjadi panduan untuk mengenali pelanggan baru.

In [3]:
# 4. CETAK ATURAN BISNIS
aturan_bisnis = export_text(model_dt_qlde, feature_names=fitur)
print("\n=== BUSINESS RULES QLDE ===\n")
print(aturan_bisnis)


=== BUSINESS RULES QLDE ===

|--- Var7 <= 4.07
|   |--- Var1 <= 105.50
|   |   |--- Var9 <= 0.50
|   |   |   |--- Var11 <= 256.20
|   |   |   |   |--- class: 2
|   |   |   |--- Var11 >  256.20
|   |   |   |   |--- class: 3
|   |   |--- Var9 >  0.50
|   |   |   |--- Var4 <= 772.47
|   |   |   |   |--- class: 2
|   |   |   |--- Var4 >  772.47
|   |   |   |   |--- class: 5
|   |--- Var1 >  105.50
|   |   |--- Var11 <= 628.33
|   |   |   |--- Var9 <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- Var9 >  0.50
|   |   |   |   |--- class: 0
|   |   |--- Var11 >  628.33
|   |   |   |--- Var9 <= 0.50
|   |   |   |   |--- class: 3
|   |   |   |--- Var9 >  0.50
|   |   |   |   |--- class: 5
|--- Var7 >  4.07
|   |--- Var4 <= 1280.95
|   |   |--- Var11 <= 281.62
|   |   |   |--- Var9 <= 0.50
|   |   |   |   |--- class: 3
|   |   |   |--- Var9 >  0.50
|   |   |   |   |--- class: 4
|   |   |--- Var11 >  281.62
|   |   |   |--- Var8 <= 59.50
|   |   |   |   |--- class: 1
|   |   |   |--- Var8 

### Cara Membaca Aturan di Atas:
Sama seperti sebelumnya, baca dari atas ke bawah. 
Misalnya untuk pelanggan baru, cek apakah nilai **Var7 <= 4.07**? Jika Ya, lanjut cek **Var1 <= 105.50**, lalu cek **Var9**, dan seterusnya sampai menemukan kesimpulan `class: ...` (Cluster pelanggan tersebut).

## 5. Menyimpan Model
Kita akan menyimpan model ini ke dalam *file* `.pkl` dengan nama khusus `_qlde` agar tidak tertukar dengan model dari *Standard K-Means*.

In [4]:
# 5. EXPORT MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)

# Simpan model dengan akhiran _qlde
joblib.dump(model_dt_qlde, '../models/model_dt_classification_qlde.pkl')
print("\n[SUCCESS] Model Decision Tree diekspor ke '../models/model_dt_classification_qlde.pkl'")


[SUCCESS] Model Decision Tree diekspor ke '../models/model_dt_classification_qlde.pkl'
